# Boltz-2 Validation — PDFA RBX1 Designs

Runs Boltz-2 complex predictions on the 14 designs from **Protein Design for Africa (PDFA)**  
using the same RBX1 target and settings used for our GLMN/CUL1_WHB designs — apples-to-apples comparison.

**Input:** `boltz_inputs/` folder (14 YAML files, binder chain A + RBX1 chain B)  
**Output:** iptm, ptm, plddt, complex_plddt per design + ranked comparison CSV  

**RBX1 target:** `MAAAMDVDTPSGTNSGAGKKRFEVKKWNAVALWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHCISRWLKTRQVCPLDNREWEFQKYGH` (108 aa, same as GLMN/CUL1_WHB runs)

In [ ]:
# ============================================================
# 📁  Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ⚙️ Set this to the folder where you uploaded the boltz_inputs/ zip
DRIVE_DIR = '/content/drive/MyDrive/adaptyv_competition'

import os
YAML_DIR  = f'{DRIVE_DIR}/pdfa_boltz_inputs'
BOLTZ_OUT = f'{DRIVE_DIR}/pdfa_boltz_outputs'
FINAL_OUT = f'{DRIVE_DIR}/pdfa_boltz_results'

for d in [YAML_DIR, BOLTZ_OUT, FINAL_OUT]:
    os.makedirs(d, exist_ok=True)

print(f'Drive mounted. Outputs will go to: {DRIVE_DIR}')

In [ ]:
# ============================================================
# 📤  Upload YAML inputs
# Upload the boltz_inputs/ folder as a zip, or copy from Drive.
# Option A: upload zip directly here
# Option B: if already on Drive, set YAML_DIR above and skip this cell
# ============================================================
import zipfile, shutil, glob

# Option A — upload zip
from google.colab import files
print('Select boltz_inputs.zip to upload (or skip if already on Drive)')
uploaded = files.upload()

for fname, data in uploaded.items():
    zip_path = f'/content/{fname}'
    with open(zip_path, 'wb') as f:
        f.write(data)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall('/content/boltz_inputs_raw')
    # Copy all yamls to YAML_DIR
    for yf in glob.glob('/content/boltz_inputs_raw/**/*.yaml', recursive=True):
        shutil.copy(yf, YAML_DIR)
    print(f'Extracted to {YAML_DIR}')

yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
print(f'\nYAML files ready: {len(yaml_files)}')
for yf in yaml_files:
    print(f'  {os.path.basename(yf)}')

In [ ]:
# ============================================================
# 🛠️  Install Boltz-2
# ============================================================
import subprocess

try:
    import boltz
    print(f'Boltz already installed: {boltz.__version__}')
except ImportError:
    print('Installing boltz...')
    r = subprocess.run('pip install -q boltz', shell=True, capture_output=True, text=True)
    print('Installed' if r.returncode == 0 else r.stderr[-500:])

# Pre-download Boltz-2 weights (cached after first run)
r = subprocess.run('boltz download', shell=True, capture_output=True, text=True)
print(r.stdout[-200:] if r.stdout else 'Weights ready')

In [ ]:
# ============================================================
# 🔬  Debug: test one prediction first to catch errors early
# ============================================================
import os, glob, subprocess

yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
assert yaml_files, f'No YAML files found in {YAML_DIR} — check upload step'

# Check boltz is callable
r = subprocess.run('boltz --help', shell=True, capture_output=True, text=True)
print('boltz CLI:', 'OK' if r.returncode == 0 else 'NOT FOUND — re-run install cell')

# Check GPU
r = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'NOT AVAILABLE — change runtime to GPU (Runtime > Change runtime type)')

# Dry-run first design with full error output visible
test_yaml = yaml_files[0]
test_name = os.path.basename(test_yaml).replace('.yaml', '')
test_out  = f'/content/boltz_test_{test_name}'
os.makedirs(test_out, exist_ok=True)

print(f'\nTest prediction: {test_name}')
cmd = (
    f'boltz predict {test_yaml} --out_dir {test_out} '
    f'--accelerator gpu '
    f'--recycling_steps 3 '
    f'--sampling_steps 200 '
    f'--diffusion_samples 1 '
    f'--no_kernels '
    f'--num_workers 2'
)
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
if r.returncode == 0:
    print('Test PASSED — proceeding with all 14 designs')
else:
    print('Test FAILED — error output:')
    print(r.stdout[-1500:])
    print(r.stderr[-1500:])
    raise RuntimeError('Fix the error above before running the full batch')

# ============================================================
# 🔬  Run all 14 designs
# ============================================================
print(f'\nRunning Boltz-2 on {len(yaml_files)} designs...\n')

for i, yf in enumerate(yaml_files):
    name    = os.path.basename(yf).replace('.yaml', '')
    out_dir = f'{BOLTZ_OUT}/{name}'

    done = glob.glob(f'{out_dir}/**/confidence_*.json', recursive=True)
    if done:
        print(f'  [{i+1:02d}/{len(yaml_files)}] {name} — already scored, skipping')
        continue

    os.makedirs(out_dir, exist_ok=True)
    cmd = (
        f'boltz predict {yf} --out_dir {out_dir} '
        f'--accelerator gpu '
        f'--recycling_steps 3 '
        f'--sampling_steps 200 '
        f'--diffusion_samples 1 '
        f'--no_kernels '
        f'--num_workers 2'
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  [{i+1:02d}/{len(yaml_files)}] {name} — OK')
    else:
        print(f'  [{i+1:02d}/{len(yaml_files)}] {name} — FAIL')
        print(r.stderr[-400:])

print('\nAll predictions complete.')

In [ ]:
# ============================================================
# 📊  Collect scores and rank
# ============================================================
import json, glob, csv as _csv

yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))

# Parse sequences from YAMLs
import yaml
def get_binder_seq(yaml_path):
    with open(yaml_path) as f:
        d = yaml.safe_load(f)
    return d['sequences'][0]['protein']['sequence']

results = []
for yf in yaml_files:
    name = os.path.basename(yf).replace('.yaml', '')
    conf_files = glob.glob(f'{BOLTZ_OUT}/{name}/**/confidence_*.json', recursive=True)
    if not conf_files:
        print(f'  WARNING: no confidence file for {name}')
        continue
    with open(conf_files[0]) as f:
        conf = json.load(f)
    seq = get_binder_seq(yf)
    results.append({
        'design':          name,
        'source':          'PDFA',
        'boltz_iptm':      conf.get('iptm', 0.0),
        'boltz_ptm':       conf.get('ptm', 0.0),
        'boltz_plddt':     conf.get('complex_plddt', conf.get('plddt', 0.0)),
        'boltz_iplddt':    conf.get('interface_plddt', conf.get('complex_iplddt', 0.0)),
        'length':          len(seq),
        'sequence':        seq,
    })

results.sort(key=lambda r: -r['boltz_iptm'])

print(f'\nScored {len(results)} PDFA designs with Boltz-2')
print(f'\n{"Design":<45} {"iptm":>6} {"ptm":>6} {"plddt":>7} {"len":>5}')
print('-' * 72)
for r in results:
    print(f"{r['design']:<45} {r['boltz_iptm']:>6.3f} {r['boltz_ptm']:>6.3f} {r['boltz_plddt']:>7.3f} {r['length']:>5}")

In [ ]:
# ============================================================
# 🏆  Side-by-side comparison: PDFA (Boltz) vs Steamulater (Boltz)
# ============================================================

# Our top GLMN designs from master_sequences.csv (pre-scored)
our_designs = [
    {'design': 'GLMN_T0.1_s11',  'source': 'Steamulater', 'boltz_iptm': 0.8869, 'boltz_ptm': 0.9091, 'boltz_plddt': 0.7534, 'length': 247},
    {'design': 'GLMN_T0.3_s12',  'source': 'Steamulater', 'boltz_iptm': 0.8816, 'boltz_ptm': 0.9037, 'boltz_plddt': 0.7953, 'length': 247},
    {'design': 'GLMN_T0.3_s8',   'source': 'Steamulater', 'boltz_iptm': 0.8781, 'boltz_ptm': 0.9036, 'boltz_plddt': 0.7770, 'length': 247},
    {'design': 'GLMN_T0.2_s14',  'source': 'Steamulater', 'boltz_iptm': 0.8790, 'boltz_ptm': 0.9011, 'boltz_plddt': 0.7593, 'length': 247},
]

# Combine and sort
all_results = our_designs + results
all_results.sort(key=lambda r: -r['boltz_iptm'])

print('=== COMBINED RANKING — Boltz-2 iptm (same predictor, same RBX1 target) ===')
print(f'\n{"#":<4} {"Design":<45} {"Source":<14} {"iptm":>6} {"ptm":>6} {"plddt":>7} {"len":>5}')
print('-' * 90)
for i, r in enumerate(all_results, 1):
    print(f"{i:<4} {r['design']:<45} {r['source']:<14} {r['boltz_iptm']:>6.3f} {r['boltz_ptm']:>6.3f} {r['boltz_plddt']:>7.3f} {r['length']:>5}")

In [ ]:
# ============================================================
# 💾  Export comparison CSV + download
# ============================================================
import csv as _csv
from google.colab import files as colab_files

csv_path = f'{FINAL_OUT}/pdfa_vs_steamulater_boltz.csv'
fields = ['design', 'source', 'boltz_iptm', 'boltz_ptm', 'boltz_plddt', 'length']

with open(csv_path, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
    w.writeheader()
    w.writerows(all_results)

print(f'Saved: {csv_path}')
colab_files.download(csv_path)